# Quick PINN Tutorial with PyTorch 

## A Simple PINN Model

**Do without LLM**

In [3]:
import torch
import torch.nn as nn
import torch
import torch.nn as nn

class SimplePINN(nn.Module):
    def __init__(self):
        super(SimplePINN, self).__init__()
        # Takes (x, t) which is size (N, 2) as input and maps to u

        # Feel free to play around with the architecture!
        self.net = nn.Sequential(
            nn.Linear(2, 64), # Linear regression layer
            nn.Tanh(), # Non-linear activation function
            nn.Linear(64, 64),
            nn.Tanh(), # Use Tanh instead of ReLU for better performance in PINNs
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x, t):
        # Concatenate x and t into a single input tensor of shape (N, 2)
        X = torch.cat([x, t], dim=1)
        return self.net(X)

model = SimplePINN()
print(model)

SimplePINN(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


## Training Loop

Burger's equation is 

$$
\frac{\partial u}{\partial t} + u \frac{\partial u }{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}
$$

If you reorder the equation:

$$
\frac{\partial u}{\partial t} + u \frac{\partial u }{\partial x} - \nu \frac{\partial^2 u}{\partial x^2} = 0
$$

Suppose $\hat{u}$ is the *model's* output. If our PINN were to actually work, we expect that 

$$
\frac{\partial \hat{u}}{\partial t} + \hat{u} \frac{\partial \hat{u} }{\partial x} - \nu \frac{\partial^2 \hat{u}}{\partial x^2} \rightarrow 0
$$

As our model gets more and more trained.

`compute_physics_loss()` literally just takes `u`, which is our $\hat{u}$ (again, the model's output), and see if that second equation goes to zero. 

For convenience purposes, we'll square the result of `pde_residual = u_t + u * u_x - nu * u_xx` to get a *magnitude*. If this thing tends to zero, then the square of that should also equal zero. So you'll need to return something like `pde_residual ** 2`. This is what `compute_physics_loss()` does. 

In [4]:
def compute_physics_loss(model, x, t, nu=0.01):
    # Ensure gradients can be computed with respect to x and t
    x.requires_grad_(True)
    t.requires_grad_(True)
    
    # Forward pass
    u = model(x, t)

    # torch.autograd() handles differentiation through AUTOMATIC DIFFERENTIATION!
    
    # First-order derivatives
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    
    # Second-order derivative
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    
    # Burgers' Equation residual: 
    # TODO: Implement the PDE residual for Burgers' equation 
    pde_residual = u_t + u * u_x - nu * u_xx
    return pde_residual**2
    # We want this residual to be as close to 0 as possible
    raise NotImplementedError("Implement the PDE residual for Burgers' equation.")

Next, we have to calculate the initial condition loss and the boundary condition losses. In a nutshell, it's simply the sum of the MSE of the model prediction on the initial/boundary condition and the true value. 

In [5]:
def compute_data_loss(model, x_ic, t_ic, u_ic_target):
    # 1. Compute Data Losses (Initial Condition and Boundary Conditions)
    u_ic_pred = model(x_ic, t_ic)
    loss_ic = mse_loss(u_ic_pred, u_ic_target)
    
    u_bc_left_pred = model(x_bc_left, t_bc)
    u_bc_right_pred = model(x_bc_right, t_bc)
    loss_bc = mse_loss(u_bc_left_pred, u_bc_target) + mse_loss(u_bc_right_pred, u_bc_target)

    return loss_ic + loss_bc

Now that we have a way to quantify Physics loss, all we do is add a term (alongside an importance factor) $\lambda$. 

That is, the PINN loss (that we'll use) is:

$$
\lambda_1 (\text{MSE}) + \lambda_2 (\text{Physics Loss}) + \lambda_3 (\text{Initial Condition Loss})
$$

In [16]:
def PINNLoss(model, x, t, y_true, x_ic, t_ic, u_ic_target, lambda1=0.1, lambda2=0.1, lambda3=0.1):
    # Forward pass
    y_pred = model(x, t)
    
    # Compute the loss (Mean Squared Error)
    model_loss = nn.MSELoss()(y_pred, y_true)
    phys_loss = compute_physics_loss(model, x, t)
    data_loss = compute_data_loss(model, x_ic, t_ic, u_ic_target)

    # TODO: combine the MSE loss, data loss physics loss with a weighting factor
    total_loss = lambda1 * model_loss + lambda2 * phys_loss + lambda3 * data_loss
    return total_loss
    raise NotImplementedError("Combine data_loss and phys_loss with a weighting factor.")

In [17]:
# Create training points for Initial Conditions (t = 0, x uniformly sampled between -1 and 1)
x_ic = torch.linspace(-1, 1, 100).view(-1, 1)
t_ic = torch.zeros_like(x_ic)
u_ic_target = -torch.sin(torch.pi * x_ic)

# Create training points for Boundary Conditions (x = -1 or x = 1, t uniformly sampled)
t_bc = torch.rand(100, 1)
x_bc_left = -torch.ones_like(t_bc)
x_bc_right = torch.ones_like(t_bc)
u_bc_target = torch.zeros_like(t_bc)

# Random interior collocation points for physics evaluation
x_collocation = (torch.rand(2000, 1) * 2) - 1  # Range [-1, 1]
t_collocation = torch.rand(2000, 1)             # Range [0, 1]

In [18]:
model = SimplePINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # An optimizer like gradient descesnt
mse_loss = nn.MSELoss()

for epoch in range(5000):
    optimizer.zero_grad()
    
    # Total Composite Loss
    total_loss = PINNLoss(model, x_collocation, t_collocation, None, x_ic, t_ic, u_ic_target, lambda1=0.1, lambda2=0.1, lambda3=0.1)
    
    total_loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Total Loss: {total_loss.item():.5f}")


AttributeError: 'NoneType' object has no attribute 'size'

# Your Task

**Do with/without LLM**

We'll use a very well known benchmark called the viscous Burger's benchmark. The initial conditions are:

$u(x,0) = −\sin(\pi x) \qquad \text{and} \qquad \nu = 0.01/\pi$

Train your PINN based on those initial conditions. Next, shrink $\nu$ from $0.001/\pi$ to $0.0001/\pi$. This will make the shock very very steep, which makes PINNs struggle!